# Stage 0 Contract and Legacy-Quarantine Gate

Validates the authoritative data/baseline contracts, reusable agent registry, and artifact classification without running preprocessing or modeling.

In [1]:
from pathlib import Path
import json
import pandas as pd
import yaml

ROOT = Path.cwd().resolve()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "data").exists(), "Could not locate project data directory"
REPO_ROOT = ROOT.parent
DATA_CONTRACT = ROOT / "configs" / "data_contract_v1.json"
BASELINE_PROTOCOL = ROOT / "configs" / "baseline_protocol_v1.json"
REGISTRY = REPO_ROOT / ".codex" / "agents" / "registry.yaml"
INVENTORY = ROOT / "reports" / "repository_audit" / "artifact_inventory_v1.csv"

In [2]:
data = json.loads(DATA_CONTRACT.read_text(encoding="utf-8"))
baseline = json.loads(BASELINE_PROTOCOL.read_text(encoding="utf-8"))
registry = yaml.safe_load(REGISTRY.read_text(encoding="utf-8"))
inventory = pd.read_csv(INVENTORY)

assert data["canonical_orientation"] == "LPS"
assert data["intermediate_resampling_mm"] == [0.5, 0.5, 0.5]
assert data["fixed_fov_mm"] == [200.0, 200.0, 200.0]
assert data["final_shape"] == [256, 256, 256]
assert data["final_spacing_mm"] == [0.78125, 0.78125, 0.78125]
assert data["bone_channels"] == ["femur", "tibia", "patella", "fibula"]
assert data["expected_ready_counts"] == {"healthy": 58, "fractured": 13, "total": 71}
assert {"VSD_z057_Left", "VSD_z057_Right"} <= set(data["exclusions"])

assert baseline["seed"] == 42 and baseline["n_folds"] == 5
assert baseline["fold_assignment"]["method"] == "StratifiedGroupKFold"
assert baseline["encoder"]["policy"] == "per_fold_train_subjects_only"
assert baseline["shared_frontend"]["target"] == "four_channel_binary_occupancy"
assert baseline["shared_frontend"]["freeze_for_decoder_comparison"] is True
assert baseline["normalization"] == {"type": "GroupNorm", "groups": 8}
assert baseline["loss"] == {"bce_weight": 0.5, "soft_dice_weight": 0.5}
assert baseline["output_channels"] == data["bone_channels"]
assert baseline["forbidden"]["fold_ids"] == [5]
assert {"binary_union", "tsdf"} <= set(baseline["forbidden"]["targets"])

roles = registry["roles"]
role_ids = {role["id"] for role in roles}
assert role_ids == {"orchestrator", *list("ABCDEFGHIJKL"), "N"}
assert "M" not in role_ids
assert len(role_ids) == len(roles)
for role in roles:
    for dep in role.get("dependencies", []):
        assert dep in role_ids, f"{role['id']} has unknown dependency {dep}"

allowed_classes = {"authoritative", "reusable", "provisional", "legacy", "smoke-only", "generated"}
assert set(inventory["classification"].dropna()) <= allowed_classes
assert inventory["classification"].notna().all()
assert inventory["path"].is_unique
print("STAGE 0 CONTRACT STATIC GATE PASS")

STAGE 0 CONTRACT STATIC GATE PASS


## Interpretation

This is a static configuration gate. It does not certify any sample, HPC output, FCMAE checkpoint, or decoder result. Those require their own evidence and Agent N review.